# Ανάλυση Ιατρικών Εικόνων DICOM στην Python

### Ιστογράμματα, Στατιστική Ανάλυση και Θεμελιώδεις Έννοιες

---

**Σε ποιον απευθύνεται αυτό το notebook:** Φοιτητές χωρίς προηγούμενη εμπειρία στην ανάλυση ιατρικών εικόνων.

**Τι θα μάθετε:**
- Πώς να «διαβάζετε» μια ιατρική εικόνα ποσοτικά, όχι μόνο οπτικά.
- Πώς λειτουργούν τα ιστογράμματα και τι μας λένε για τους ιστούς.
- Πώς εκτιμούμε ποιότητα εικόνας (αντίθεση, φωτεινότητα, θόρυβος, SNR).
- Πώς αναλύουμε χωρικές διαφορές μέσα στην εικόνα.
- Πώς συντάσσουμε μια ολοκληρωμένη αναφορά ποιότητας.

**Προαπαιτούμενα του κώδικα:** Πριν τρέξετε τα παρακάτω κελιά πρέπει να έχει εκτελεστεί ένα προηγούμενο τμήμα του μαθήματος που φορτώνει τη μεταβλητή `dicom_data` (με `pydicom.dcmread(...)`) και κάνει τα imports `numpy as np`, `pandas as pd`, `matplotlib.pyplot as plt`, `json`.

> 💡 **Παιδαγωγική σημείωση:** Όλος ο κώδικας εδώ ακολουθεί ένα μοτίβο: *(α)* υπολογίζουμε ποσοτικά μεγέθη από τα pixel, *(β)* τα οπτικοποιούμε, *(γ)* τα ερμηνεύουμε. Προσπαθήστε σε κάθε ενότητα να αναγνωρίζετε αυτά τα τρία βήματα.


## Ενότητα 26 — Κατανόηση Ιστογραμμάτων Εικόνας

### Τι είναι μια ιατρική εικόνα στον υπολογιστή;

Πριν μιλήσουμε για ιστογράμματα, ας ξεκαθαρίσουμε τι «βλέπει» πραγματικά ο υπολογιστής. Μια εικόνα DICOM (CT, MRI, mammography κ.λπ.) είναι **ένας πίνακας αριθμών**. Κάθε θέση του πίνακα αντιστοιχεί σε ένα **pixel** (εικονοστοιχείο) και ο αριθμός εκεί λέγεται **ένταση** (intensity). Στην ακτινολογία, αυτή η ένταση συνδέεται με φυσικά μεγέθη — π.χ. στο CT αντιστοιχεί σε **Hounsfield Units** (πυκνότητα ιστού), στο MRI εξαρτάται από το είδος της ακολουθίας (T1, T2, DWI κ.λπ.).

Άρα όταν λέμε «ανάλυση εικόνας», στην πράξη κάνουμε στατιστική σε εκατομμύρια αριθμούς.

### Τι είναι το ιστόγραμμα;

Το **ιστόγραμμα** είναι ένα διάγραμμα που δείχνει **πόσα pixel έχουν την κάθε τιμή έντασης**. Δηλαδή:

- Στον **οριζόντιο άξονα** (x): οι τιμές έντασης (από τη μικρότερη στη μεγαλύτερη).
- Στον **κατακόρυφο άξονα** (y): πόσες φορές εμφανίζεται η κάθε τιμή στην εικόνα.

> 📌 **Διαισθητική αναλογία:** Φανταστείτε ότι ρίχνετε όλα τα pixel σε «κουτάκια» (bins) ανάλογα με την τιμή τους — π.χ. ένα κουτί για τιμές 0–10, ένα για 10–20, κ.ο.κ. Το ιστόγραμμα είναι απλά το ύψος αυτών των κουτιών.

### Γιατί έχει σημασία στην ιατρική;

Διαφορετικοί ιστοί έχουν διαφορετικές εντάσεις. Σε ένα CT θώρακος, για παράδειγμα, ο **αέρας** δίνει πολύ χαμηλές τιμές, το **μαλακό μόριο** μεσαίες, και το **οστό** πολύ υψηλές. Όταν δούμε ένα ιστόγραμμα με δύο ή τρεις «κορυφές» (peaks/modes), αυτό μας λέει ότι υπάρχουν στην εικόνα διακριτοί ιστοί — μια **πολυτροπική** (multimodal) κατανομή.

### Τι θα δείτε στον κώδικα παρακάτω

Η συνάρτηση `comprehensive_histogram_analysis` θα παρουσιάσει **έξι διαφορετικές αναπαραστάσεις** του ίδιου ιστογράμματος. Καθεμιά απαντά σε διαφορετική ερώτηση:

1. **Βασικό ιστόγραμμα** — Πώς κατανέμονται γενικά οι εντάσεις;
2. **Ιστόγραμμα με στατιστικές γραμμές** — Πού πέφτει η μέση τιμή, η διάμεσος, και το ±1 τυπική απόκλιση;
3. **Αθροιστικό ιστόγραμμα (CDF)** — Τι ποσοστό των pixel βρίσκεται κάτω από κάθε τιμή; (Βασικό εργαλείο για windowing/thresholding.)
4. **Ιστόγραμμα σε λογαριθμική κλίμακα** — Αν λίγα pixel έχουν ακραίες τιμές, η γραμμική κλίμακα τα «κρύβει». Ο λογάριθμος τα φέρνει στην επιφάνεια.
5. **Κανονικοποιημένο ιστόγραμμα (PDF)** — Αντί για πλήθος, βλέπουμε **πιθανότητα**. Έτσι ιστογράμματα από εικόνες διαφορετικού μεγέθους γίνονται συγκρίσιμα.
6. **Ιστόγραμμα με εκατοστημόρια** — Πού πέφτουν τα 1ο, 5ο, 25ο, 50ο, 75ο, 95ο, 99ο εκατοστημόριο. Αυτό μας βοηθά να εντοπίσουμε ακραίες τιμές (outliers) και να επιλέξουμε σωστό window/level για την προβολή.

> ⚠️ **Σημαντικό:** Ο αριθμός των bins (εδώ 50) επηρεάζει την εμφάνιση. Λίγα bins → πολύ «λεία» εικόνα, χάνουμε λεπτομέρεια. Πολλά bins → θορυβώδες ιστόγραμμα. Στις ασκήσεις θα δείτε μόνοι σας πώς αλλάζει η ερμηνεία ανάλογα με την επιλογή.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║            UNDERSTANDING IMAGE HISTOGRAMS                      ║
╚════════════════════════════════════════════════════════════════╝

A histogram shows the distribution of pixel intensities in an image.
- X-axis: Pixel intensity values
- Y-axis: Number of pixels with that intensity

Why histograms matter:
✓ Understanding image contrast
✓ Detecting image quality issues
✓ Choosing appropriate window/level settings
✓ Identifying different tissue types
✓ Preprocessing decisions (normalization, etc.)
""")

def comprehensive_histogram_analysis(dicom_obj, title=""):
    """
    Create comprehensive histogram visualizations
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object to analyze
    title : str
        Title for the analysis
    """
    
    pixel_array = dicom_obj.pixel_array.flatten()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 1. Basic Histogram
    axes[0, 0].hist(pixel_array, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0, 0].set_title('Basic Histogram', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Pixel Intensity')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Histogram with Statistics Lines
    axes[0, 1].hist(pixel_array, bins=50, color='lightcoral', alpha=0.7, edgecolor='black')
    mean_val = pixel_array.mean()
    median_val = np.median(pixel_array)
    std_val = pixel_array.std()
    
    axes[0, 1].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    axes[0, 1].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.1f}')
    axes[0, 1].axvline(mean_val + std_val, color='orange', linestyle=':', linewidth=2, label=f'+1 SD: {mean_val+std_val:.1f}')
    axes[0, 1].axvline(mean_val - std_val, color='orange', linestyle=':', linewidth=2, label=f'-1 SD: {mean_val-std_val:.1f}')
    
    axes[0, 1].set_title('Histogram with Statistics', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Pixel Intensity')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].legend(fontsize=9)
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Cumulative Histogram
    counts, bins = np.histogram(pixel_array, bins=100)
    cumulative = np.cumsum(counts)
    axes[0, 2].plot(bins[:-1], cumulative, color='darkgreen', linewidth=2)
    axes[0, 2].fill_between(bins[:-1], cumulative, alpha=0.3, color='lightgreen')
    axes[0, 2].set_title('Cumulative Histogram', fontsize=12, fontweight='bold')
    axes[0, 2].set_xlabel('Pixel Intensity')
    axes[0, 2].set_ylabel('Cumulative Frequency')
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. Log Scale Histogram
    axes[1, 0].hist(pixel_array, bins=50, color='purple', alpha=0.7, edgecolor='black')
    axes[1, 0].set_yscale('log')
    axes[1, 0].set_title('Histogram (Log Scale)', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Pixel Intensity')
    axes[1, 0].set_ylabel('Frequency (log scale)')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Normalized Histogram (Probability Density)
    axes[1, 1].hist(pixel_array, bins=50, density=True, color='teal', alpha=0.7, edgecolor='black')
    axes[1, 1].set_title('Normalized Histogram (PDF)', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Pixel Intensity')
    axes[1, 1].set_ylabel('Probability Density')
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Percentile Analysis
    percentiles = [1, 5, 25, 50, 75, 95, 99]
    percentile_values = np.percentile(pixel_array, percentiles)
    
    axes[1, 2].hist(pixel_array, bins=50, color='goldenrod', alpha=0.5, edgecolor='black')
    for p, val in zip(percentiles, percentile_values):
        axes[1, 2].axvline(val, color='red', linestyle='--', linewidth=1, alpha=0.7)
        axes[1, 2].text(val, axes[1, 2].get_ylim()[1]*0.9, f'{p}%', 
                       rotation=90, fontsize=8, va='top')
    
    axes[1, 2].set_title('Histogram with Percentiles', fontsize=12, fontweight='bold')
    axes[1, 2].set_xlabel('Pixel Intensity')
    axes[1, 2].set_ylabel('Frequency')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.suptitle(f'Comprehensive Histogram Analysis: {title}', 
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

# Demonstrate histogram analysis
comprehensive_histogram_analysis(dicom_data, f"{dicom_data.Modality} Image")


## Ενότητα 27 — Αναλυτική Στατιστική Ανάλυση

### Γιατί χρειαζόμαστε αριθμούς-σύνοψη;

Μια εικόνα 512×512 έχει 262.144 pixel. Δεν μπορούμε να μιλάμε για κάθε ένα ξεχωριστά — χρειαζόμαστε **συνοπτικά μέτρα** που περιγράφουν την κατανομή σε λίγους αριθμούς. Αυτά τα μέτρα είναι το ABC της ανάλυσης: μια εικόνα μπορεί να συγκριθεί με μια άλλη, ένας ιστός με έναν άλλον, μια εξέταση με μια προηγούμενη.

### Οι τέσσερις οικογένειες μέτρων

#### 1. Μέτρα κεντρικής τάσης — *Πού «κάθεται» η κατανομή;*

- **Μέση τιμή (mean, μ):** Το άθροισμα διά το πλήθος. Ευαίσθητη σε ακραίες τιμές.
- **Διάμεσος (median):** Η μεσαία τιμή όταν τα pixel ταξινομηθούν. **Ανθεκτική** σε outliers — γι' αυτό προτιμάται όταν υπάρχει θόρυβος.
- **Επικρατούσα τιμή (mode):** Η πιο συχνή τιμή. Σε ιατρικές εικόνες συχνά αντιστοιχεί στον κυρίαρχο ιστό ή στο φόντο.

> 🧠 **Κανόνας του φοιτητή:** Αν η μέση τιμή και η διάμεσος διαφέρουν σημαντικά, η κατανομή σας **δεν είναι συμμετρική**.

#### 2. Μέτρα διασποράς — *Πόσο «απλωμένη» είναι η κατανομή;*

- **Τυπική απόκλιση (std, σ):** Πόσο μακριά, κατά μέσο όρο, βρίσκονται τα pixel από τη μέση τιμή. Σε ιατρικές εικόνες αντικατοπτρίζει ταυτόχρονα την **ετερογένεια ιστών** ΚΑΙ τον **θόρυβο**.
- **Διακύμανση (variance, σ²):** Το τετράγωνο της τυπικής απόκλισης.
- **Εύρος (range):** max − min. Πολύ ευαίσθητο — αρκεί ένα κακό pixel για να το «τινάξει στον αέρα».
- **Ενδοτεταρτημοριακό εύρος (IQR):** Q3 − Q1. **Ανθεκτικό** μέτρο διασποράς, ισοδύναμο της διαμέσου.

#### 3. Μέτρα σχήματος — *Πώς «μοιάζει» η κατανομή;*

- **Ασυμμετρία (skewness):** Πόσο γέρνει η κατανομή προς τη μία πλευρά.
  - skew ≈ 0 → συμμετρική (όπως κανονική κατανομή)
  - skew > 0 → δεξιά ασυμμετρία (μακριά ουρά προς τα δεξιά, λιγοστά λαμπρά pixel)
  - skew < 0 → αριστερή ασυμμετρία
- **Κύρτωση (kurtosis):** Πόσο «μυτερή» ή «βαριά στις ουρές» είναι.
  - kurtosis > 0 → πιο μυτερή κορυφή και βαρύτερες ουρές (περισσότερα ακραία pixel)
  - kurtosis < 0 → πιο πλατιά κατανομή

#### 4. Μέτρα ποιότητας

- **Συντελεστής μεταβλητότητας (CV = σ/μ × 100%):** Σχετική διασπορά. Επιτρέπει σύγκριση μεταξύ εικόνων με διαφορετικές κλίμακες.
- **SNR (Signal-to-Noise Ratio):** Στην απλή του μορφή, μ/σ. Όσο μεγαλύτερο, τόσο «καθαρότερη» η εικόνα.

### Τι κάνει ο κώδικας

Δύο συναρτήσεις:

1. **`comprehensive_statistical_analysis`** — Υπολογίζει όλα τα παραπάνω σε ένα dictionary.
2. **`display_statistics_report`** — Παρουσιάζει τα αποτελέσματα σε καλά μορφοποιημένο πίνακα και προσθέτει **αυτόματη ερμηνεία** (π.χ. «Right-skewed»).

> 📌 Παρατηρήστε ότι η συνάρτηση επιστρέφει dictionary. Αυτή είναι καλή πρακτική: ο κώδικας που υπολογίζει διαχωρίζεται από τον κώδικα που εμφανίζει — μπορείτε να ξαναχρησιμοποιήσετε τα νούμερα σε άλλη ανάλυση χωρίς να ξανατρέξετε τα ίδια.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              DETAILED STATISTICAL ANALYSIS                     ║
╚════════════════════════════════════════════════════════════════╝

Statistical measures help us understand:
✓ Central tendency (mean, median, mode)
✓ Spread/dispersion (standard deviation, variance, range)
✓ Distribution shape (skewness, kurtosis)
✓ Outliers and extreme values
✓ Image quality and consistency
""")

def comprehensive_statistical_analysis(dicom_obj):
    """
    Perform comprehensive statistical analysis on DICOM image
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object to analyze
    
    Returns:
    --------
    dict : Dictionary containing all statistical measures
    """
    
    pixel_array = dicom_obj.pixel_array.flatten()
    
    # Basic statistics
    stats = {
        'count': len(pixel_array),
        'mean': np.mean(pixel_array),
        'median': np.median(pixel_array),
        'mode': float(np.bincount(pixel_array.astype(int)).argmax()) if pixel_array.max() < 10000 else 'N/A',
        'std': np.std(pixel_array),
        'variance': np.var(pixel_array),
        'min': np.min(pixel_array),
        'max': np.max(pixel_array),
        'range': np.ptp(pixel_array),  # peak to peak (max - min)
    }
    
    # Percentiles
    percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
    for p in percentiles:
        stats[f'percentile_{p}'] = np.percentile(pixel_array, p)
    
    # Quartiles and IQR
    q1 = np.percentile(pixel_array, 25)
    q3 = np.percentile(pixel_array, 75)
    stats['q1'] = q1
    stats['q3'] = q3
    stats['iqr'] = q3 - q1  # Interquartile range
    
    # Distribution shape
    from scipy import stats as scipy_stats
    stats['skewness'] = scipy_stats.skew(pixel_array)
    stats['kurtosis'] = scipy_stats.kurtosis(pixel_array)
    
    # Coefficient of variation
    stats['cv'] = (stats['std'] / stats['mean']) * 100 if stats['mean'] != 0 else 0
    
    # Signal to Noise Ratio (simplified)
    stats['snr'] = stats['mean'] / stats['std'] if stats['std'] != 0 else 0
    
    return stats

def display_statistics_report(stats_dict, dicom_obj):
    """
    Display a formatted statistics report
    """
    
    print("\n" + "="*70)
    print("📊 COMPREHENSIVE STATISTICAL REPORT")
    print("="*70)
    
    print(f"\n{'IMAGE INFORMATION':^70}")
    print("-"*70)
    print(f"{'Modality:':<30} {dicom_obj.Modality}")
    print(f"{'Image Size:':<30} {dicom_obj.Rows} x {dicom_obj.Columns}")
    print(f"{'Total Pixels:':<30} {stats_dict['count']:,}")
    
    print(f"\n{'CENTRAL TENDENCY MEASURES':^70}")
    print("-"*70)
    print(f"{'Mean:':<30} {stats_dict['mean']:>20.2f}")
    print(f"{'Median:':<30} {stats_dict['median']:>20.2f}")
    print(f"{'Mode:':<30} {stats_dict['mode']:>20}")
    
    print(f"\n{'DISPERSION MEASURES':^70}")
    print("-"*70)
    print(f"{'Standard Deviation:':<30} {stats_dict['std']:>20.2f}")
    print(f"{'Variance:':<30} {stats_dict['variance']:>20.2f}")
    print(f"{'Range (Max - Min):':<30} {stats_dict['range']:>20.2f}")
    print(f"{'Interquartile Range (IQR):':<30} {stats_dict['iqr']:>20.2f}")
    print(f"{'Coefficient of Variation:':<30} {stats_dict['cv']:>19.2f}%")
    
    print(f"\n{'EXTREME VALUES':^70}")
    print("-"*70)
    print(f"{'Minimum:':<30} {stats_dict['min']:>20.2f}")
    print(f"{'Maximum:':<30} {stats_dict['max']:>20.2f}")
    print(f"{'1st Percentile:':<30} {stats_dict['percentile_1']:>20.2f}")
    print(f"{'99th Percentile:':<30} {stats_dict['percentile_99']:>20.2f}")
    
    print(f"\n{'QUARTILES':^70}")
    print("-"*70)
    print(f"{'Q1 (25th percentile):':<30} {stats_dict['q1']:>20.2f}")
    print(f"{'Q2 (50th percentile/Median):':<30} {stats_dict['median']:>20.2f}")
    print(f"{'Q3 (75th percentile):':<30} {stats_dict['q3']:>20.2f}")
    
    print(f"\n{'DISTRIBUTION SHAPE':^70}")
    print("-"*70)
    print(f"{'Skewness:':<30} {stats_dict['skewness']:>20.4f}")
    skew_interpretation = "Right-skewed" if stats_dict['skewness'] > 0 else "Left-skewed" if stats_dict['skewness'] < 0 else "Symmetric"
    print(f"{'  → Interpretation:':<30} {skew_interpretation:>20}")
    
    print(f"{'Kurtosis:':<30} {stats_dict['kurtosis']:>20.4f}")
    kurt_interpretation = "Heavy-tailed" if stats_dict['kurtosis'] > 0 else "Light-tailed" if stats_dict['kurtosis'] < 0 else "Normal"
    print(f"{'  → Interpretation:':<30} {kurt_interpretation:>20}")
    
    print(f"\n{'QUALITY METRICS':^70}")
    print("-"*70)
    print(f"{'Signal-to-Noise Ratio (SNR):':<30} {stats_dict['snr']:>20.2f}")
    
    print("\n" + "="*70)
    
    # Interpretation guide
    print(f"\n{'📖 INTERPRETATION GUIDE':^70}")
    print("-"*70)
    print("""
    Skewness:
      • = 0  : Symmetric distribution
      • > 0  : Right-skewed (tail extends right, more low values)
      • < 0  : Left-skewed (tail extends left, more high values)
    
    Kurtosis:
      • = 0  : Normal distribution (mesokurtic)
      • > 0  : Heavy-tailed, more outliers (leptokurtic)
      • < 0  : Light-tailed, fewer outliers (platykurtic)
    
    Coefficient of Variation:
      • Low (<15%) : Low variability relative to mean
      • High (>30%) : High variability relative to mean
    """)

# Perform and display statistical analysis
stats_result = comprehensive_statistical_analysis(dicom_data)
display_statistics_report(stats_result, dicom_data)


## Ενότητα 28 — Συγκριτική Οπτικοποίηση Στατιστικών

### Γιατί δεν αρκούν οι αριθμοί;

Στην προηγούμενη ενότητα παραγάγαμε δεκάδες αριθμούς. Αλλά οι άνθρωποι δεν διαβάζουμε καλά πίνακες με 30+ νούμερα — βλέπουμε πολύ καλύτερα **σχήματα και σχέσεις**. Σε αυτή την ενότητα θα δείτε τέσσερα κλασικά διαγράμματα που «μεταφράζουν» τη στατιστική σε εικόνα.

### Τα τέσσερα διαγράμματα του κώδικα

#### 1. Box plot (διάγραμμα κουτιού)
Πέντε αριθμοί σε ένα σχήμα: minimum, Q1, διάμεσος, Q3, maximum.

- Το **κουτί** καλύπτει το IQR (το μεσαίο 50% των pixel).
- Η **κόκκινη γραμμή** μέσα είναι η διάμεσος.
- Τα **whiskers** (μουστάκια) εκτείνονται μέχρι περίπου τα ακραία «κανονικά» σημεία.
- Σημεία πέρα από αυτά → **outliers**.

> Χρήση: Γρήγορη αναγνώριση ακραίων τιμών και ασυμμετρίας.

#### 2. Violin plot (διάγραμμα βιολιού)
Συνδυάζει box plot + ιστόγραμμα. Το πάχος του «βιολιού» σε κάθε ύψος δείχνει **πυκνότητα pixel**. Όπου το βιολί φαρδαίνει, εκεί συγκεντρώνονται περισσότερα pixel.

> Χρήση: Όταν θέλετε να φανεί η μορφή της κατανομής, όχι μόνο τα στατιστικά της.

#### 3. Q-Q plot (Quantile-Quantile)
**Το πιο σημαντικό διαγνωστικό διάγραμμα** της ενότητας. Συγκρίνει τα ποσοστημόρια των δεδομένων μας με αυτά μιας **κανονικής κατανομής**.

- Αν τα σημεία πέφτουν πάνω στην ευθεία αναφοράς → η κατανομή είναι περίπου κανονική.
- Αν τα σημεία αποκλίνουν, ειδικά στις άκρες → η κατανομή **δεν είναι κανονική** (συνηθισμένο για ιατρικές εικόνες, που έχουν συχνά πολυτροπικές κατανομές με φόντο, ιστούς, οστά).

> ⚠️ Πολλές στατιστικές μέθοδοι (π.χ. t-test) προϋποθέτουν κανονικότητα. Πάντα ελέγχετε με Q-Q plot πριν εφαρμόσετε.

#### 4. Bar chart κύριων στατιστικών
Ραβδόγραμμα που δείχνει σε μία ματιά τη σχέση μέσης τιμής, διαμέσου, τυπικής απόκλισης και τεταρτημορίων. Χρήσιμο για **παρουσιάσεις** και αναφορές.

> 💡 **Tip:** Όταν παρουσιάζετε αποτελέσματα σε γιατρό ή σε επιστημονική επιτροπή, το box plot και το violin plot «μιλάνε» πιο γρήγορα από τους πίνακες.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║          COMPARATIVE STATISTICS VISUALIZATION                  ║
╚════════════════════════════════════════════════════════════════╝
""")

def visualize_statistics_comparison(dicom_obj):
    """
    Create visualizations comparing different statistical aspects
    """
    
    pixel_array = dicom_obj.pixel_array
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Box Plot
    axes[0, 0].boxplot(pixel_array.flatten(), vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightblue', color='blue'),
                       medianprops=dict(color='red', linewidth=2),
                       whiskerprops=dict(color='blue'),
                       capprops=dict(color='blue'))
    
    axes[0, 0].set_ylabel('Pixel Intensity', fontsize=12)
    axes[0, 0].set_title('Box Plot - Distribution Summary', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Add annotations
    q1, median, q3 = np.percentile(pixel_array.flatten(), [25, 50, 75])
    axes[0, 0].text(1.15, q1, f'Q1: {q1:.1f}', fontsize=10, va='center')
    axes[0, 0].text(1.15, median, f'Median: {median:.1f}', fontsize=10, va='center', color='red')
    axes[0, 0].text(1.15, q3, f'Q3: {q3:.1f}', fontsize=10, va='center')
    
    # 2. Violin Plot (using histogram approximation)
    axes[0, 1].violinplot([pixel_array.flatten()], vert=True, showmeans=True, showmedians=True)
    axes[0, 1].set_ylabel('Pixel Intensity', fontsize=12)
    axes[0, 1].set_title('Violin Plot - Distribution Density', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # 3. Q-Q Plot (Quantile-Quantile)
    from scipy import stats as scipy_stats
    theoretical_quantiles = scipy_stats.norm.ppf(np.linspace(0.01, 0.99, 100))
    sample_quantiles = np.percentile(pixel_array.flatten(), np.linspace(1, 99, 100))
    
    axes[1, 0].scatter(theoretical_quantiles, sample_quantiles, alpha=0.6, s=20)
    axes[1, 0].plot(theoretical_quantiles, 
                    theoretical_quantiles * pixel_array.std() + pixel_array.mean(),
                    'r--', linewidth=2, label='Normal distribution reference')
    axes[1, 0].set_xlabel('Theoretical Quantiles (Normal)', fontsize=12)
    axes[1, 0].set_ylabel('Sample Quantiles', fontsize=12)
    axes[1, 0].set_title('Q-Q Plot - Normality Check', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Statistical Summary Bar Chart
    stats_summary = {
        'Mean': pixel_array.mean(),
        'Median': np.median(pixel_array),
        'Std Dev': pixel_array.std(),
        'Q1': np.percentile(pixel_array, 25),
        'Q3': np.percentile(pixel_array, 75)
    }
    
    bars = axes[1, 1].bar(stats_summary.keys(), stats_summary.values(), 
                          color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8'],
                          edgecolor='black', linewidth=1.5)
    
    axes[1, 1].set_ylabel('Value', fontsize=12)
    axes[1, 1].set_title('Key Statistics Comparison', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.1f}',
                       ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

visualize_statistics_comparison(dicom_data)


## Ενότητα 29 — Στατιστική ανά Περιοχή (Region-Based Statistics)

### Το πρόβλημα του «καθολικού μέσου όρου»

Μέχρι τώρα υπολογίζαμε στατιστικά για **όλη την εικόνα**. Αλλά μια ιατρική εικόνα είναι σπάνια ομοιόμορφη! Ένας όγκος καταλαμβάνει το 5% της εικόνας — αν τον αναμίξετε με 95% υγιή ιστό, τα στατιστικά του «πνίγονται». Επιπλέον, τεχνικά artifacts (π.χ. **ανομοιογένεια πεδίου** στο MRI, *bias field*) εμφανίζονται ως χωρικές διαφορές που η καθολική μέση τιμή κρύβει εντελώς.

**Λύση:** χωρίζουμε την εικόνα σε υπο-περιοχές και υπολογίζουμε στατιστικά **σε κάθε μία ξεχωριστά**.

### Πότε μας ενδιαφέρει η ανάλυση ανά περιοχή;

- **Έλεγχος ομοιογένειας** του φόντου (background uniformity) — βασικό QC.
- **Εντοπισμός artifacts** — αν μια περιοχή έχει πολύ διαφορετική τυπική απόκλιση από τις γειτονικές, κάτι «πειράζει».
- **Ετερογένεια όγκου** — στην ογκολογία, η χωρική ετερογένεια είναι προγνωστικός παράγοντας.
- **Bias field correction** — εντοπισμός βαθμωτών μεταβολών έντασης σε MRI.

### Τι κάνει ο κώδικας

Η `analyze_image_regions` χωρίζει την εικόνα σε **πλέγμα 3×3** (9 περιοχές) και για κάθε μία υπολογίζει mean, std, min, max, median. Το αποτέλεσμα γίνεται `pandas.DataFrame` — μια εξαιρετικά βολική μορφή για περαιτέρω ανάλυση.

Στη συνέχεια η `visualize_region_statistics` παράγει:

1. **Heatmap μέσης τιμής** — Δείχνει χωρικά πού είναι «πιο φωτεινή» η εικόνα. Ομοιόμορφο χρώμα → ομοιόμορφη εικόνα. Βαθμωτό χρώμα → bias.
2. **Heatmap τυπικής απόκλισης** — Πού υπάρχει μεγαλύτερη μεταβλητότητα (συχνά εκεί όπου υπάρχουν δομές, ακμές, ιστοί).
3. **Bar chart με error bars** — Μέση τιμή ανά περιοχή με ±1 std.
4. **CV ανά περιοχή** — Σχετική μεταβλητότητα. Βοηθά στον εντοπισμό περιοχών με ασυνήθιστα υψηλή ή χαμηλή ετερογένεια.

> 📌 **Παράμετρος `grid_size`:** Στον κώδικα είναι 3 (3×3 = 9 περιοχές). Σε εικόνες υψηλής ανάλυσης ίσως θέλετε 5×5 ή 10×10. Στις ασκήσεις θα παίξετε με αυτή την παράμετρο.

> 🧠 **Σύνδεση με προχωρημένες έννοιες:** Αυτή η ιδέα — υπολογισμός χαρακτηριστικών σε τοπικά «patches» — είναι **ο πυρήνας** πολλών σύγχρονων μεθόδων: convolutional neural networks, radiomics texture features, και sliding-window analysis.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              REGION-BASED STATISTICS                           ║
╚════════════════════════════════════════════════════════════════╝

Analyzing statistics in different regions can reveal:
✓ Spatial variations in image intensity
✓ Tissue heterogeneity
✓ Image artifacts
✓ Quality assessment
""")

def analyze_image_regions(dicom_obj, grid_size=3):
    """
    Divide image into grid and analyze each region separately
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object
    grid_size : int
        Size of the grid (e.g., 3 means 3x3 = 9 regions)
    
    Returns:
    --------
    pd.DataFrame : Statistics for each region
    """
    
    img = dicom_obj.pixel_array
    rows, cols = img.shape
    
    # Calculate region sizes
    row_step = rows // grid_size
    col_step = cols // grid_size
    
    region_stats = []
    
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(12, 12))
    
    for i in range(grid_size):
        for j in range(grid_size):
            # Extract region
            r_start = i * row_step
            r_end = (i + 1) * row_step if i < grid_size - 1 else rows
            c_start = j * col_step
            c_end = (j + 1) * col_step if j < grid_size - 1 else cols
            
            region = img[r_start:r_end, c_start:c_end]
            
            # Calculate statistics
            region_stats.append({
                'Region': f'R{i+1}C{j+1}',
                'Row': i + 1,
                'Col': j + 1,
                'Mean': region.mean(),
                'Std': region.std(),
                'Min': region.min(),
                'Max': region.max(),
                'Median': np.median(region)
            })
            
            # Visualize region
            ax = axes[i, j] if grid_size > 1 else axes
            ax.imshow(region, cmap='gray')
            ax.set_title(f'R{i+1}C{j+1}\nμ={region.mean():.0f}', fontsize=9)
            ax.axis('off')
            
            # Add colored border based on mean intensity
            for spine in ax.spines.values():
                spine.set_edgecolor('red' if region.mean() > img.mean() else 'blue')
                spine.set_linewidth(3)
    
    plt.suptitle(f'{grid_size}x{grid_size} Region Analysis\nRed=Above Average, Blue=Below Average',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Create DataFrame
    df = pd.DataFrame(region_stats)
    
    return df

# Perform region analysis
print("\nPerforming 3x3 region analysis...")
region_df = analyze_image_regions(dicom_data, grid_size=3)
print("\n📊 Region Statistics Table:")
display(region_df)

# Visualize region statistics
def visualize_region_statistics(region_df):
    """
    Visualize statistics across regions
    """
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Mean intensity heatmap
    mean_grid = region_df.pivot(index='Row', columns='Col', values='Mean')
    im1 = axes[0, 0].imshow(mean_grid, cmap='hot', aspect='auto')
    axes[0, 0].set_title('Mean Intensity Heatmap', fontweight='bold')
    axes[0, 0].set_xlabel('Column')
    axes[0, 0].set_ylabel('Row')
    plt.colorbar(im1, ax=axes[0, 0])
    
    # Add values on heatmap
    for i in range(len(mean_grid)):
        for j in range(len(mean_grid.columns)):
            axes[0, 0].text(j, i, f'{mean_grid.iloc[i, j]:.0f}',
                          ha='center', va='center', color='white', fontweight='bold')
    
    # Standard deviation heatmap
    std_grid = region_df.pivot(index='Row', columns='Col', values='Std')
    im2 = axes[0, 1].imshow(std_grid, cmap='viridis', aspect='auto')
    axes[0, 1].set_title('Standard Deviation Heatmap', fontweight='bold')
    axes[0, 1].set_xlabel('Column')
    axes[0, 1].set_ylabel('Row')
    plt.colorbar(im2, ax=axes[0, 1])
    
    # Add values on heatmap
    for i in range(len(std_grid)):
        for j in range(len(std_grid.columns)):
            axes[0, 1].text(j, i, f'{std_grid.iloc[i, j]:.0f}',
                          ha='center', va='center', color='white', fontweight='bold')
    
    # Bar chart comparison
    x_pos = np.arange(len(region_df))
    axes[1, 0].bar(x_pos, region_df['Mean'], alpha=0.7, label='Mean', color='steelblue')
    axes[1, 0].errorbar(x_pos, region_df['Mean'], yerr=region_df['Std'], 
                       fmt='none', color='red', capsize=5, label='±1 Std')
    axes[1, 0].set_xlabel('Region')
    axes[1, 0].set_ylabel('Intensity')
    axes[1, 0].set_title('Mean Intensity per Region (with Std Dev)', fontweight='bold')
    axes[1, 0].set_xticks(x_pos)
    axes[1, 0].set_xticklabels(region_df['Region'], rotation=45)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Coefficient of variation
    cv = (region_df['Std'] / region_df['Mean']) * 100
    axes[1, 1].bar(region_df['Region'], cv, color='coral', edgecolor='black')
    axes[1, 1].set_xlabel('Region')
    axes[1, 1].set_ylabel('Coefficient of Variation (%)')
    axes[1, 1].set_title('Variability per Region', fontweight='bold')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].axhline(y=cv.mean(), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean CV: {cv.mean():.1f}%')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()

visualize_region_statistics(region_df)


## Ενότητα 30 — Ανάλυση Προφίλ Έντασης (Intensity Profiles)

### Τι είναι ένα προφίλ έντασης;

Φανταστείτε ότι «κόβετε» την εικόνα κατά μήκος μιας ευθείας γραμμής και κοιτάτε τις τιμές των pixel πάνω σε αυτή τη γραμμή. Αυτό είναι ένα **προφίλ έντασης**: μια μονοδιάστατη ακολουθία αριθμών που δείχνει πώς αλλάζει η ένταση κατά μήκος της γραμμής.

> 📌 **Αναλογία:** Όπως όταν ένα GPS δείχνει το **υψομετρικό προφίλ** μιας πεζοπορίας — από επίπεδο σε ανήφορο, σε κορυφή, σε κατήφορο. Εδώ το «υψόμετρο» είναι η ένταση των pixel.

### Γιατί είναι χρήσιμα;

1. **Εντοπισμός ορίων (edges):** Όπου το προφίλ αλλάζει απότομα → εκεί υπάρχει σύνορο μεταξύ ιστών.
2. **Μέτρηση αποστάσεων:** Συνδυάζοντας με το `PixelSpacing` του DICOM, μπορούμε να μετρήσουμε διαστάσεις ανατομικών δομών σε χιλιοστά.
3. **Quality control:** Αν το προφίλ φαίνεται «θολωμένο» αντί για απότομες μεταβάσεις, η εικόνα έχει χαμηλή χωρική ανάλυση ή κίνηση.
4. **Ποσοτική σύγκριση:** Πριν/μετά θεραπεία — πώς άλλαξε το προφίλ στην ίδια θέση.

### Τι κάνει ο κώδικας

Η `analyze_intensity_profiles` εξάγει **τρία προφίλ**:

- **Οριζόντιο** — η μεσαία γραμμή της εικόνας.
- **Κάθετο** — η μεσαία στήλη.
- **Διαγώνιο** — από επάνω-αριστερά προς κάτω-δεξιά.

Και στη συνέχεια υπολογίζει την **κλίση (gradient)** του οριζόντιου προφίλ. Αυτό είναι κρίσιμο: η κλίση είναι ο **ρυθμός μεταβολής** της έντασης. Όταν η κλίση είναι κοντά στο 0, βρισκόμαστε σε μια ομοιογενή περιοχή. Όταν η κλίση εκτοξεύεται (θετικά ή αρνητικά), έχουμε **ακμή** — μετάβαση από έναν ιστό σε άλλον.

```
Προφίλ:    ─────╱╲────────╱╲──────
Gradient:  ─0─▲──▼─0─0─▲──▼─0──
                ↑       ↑
              ακμές (edges)
```

> 🧠 **Σύνδεση με computer vision:** Όλοι οι κλασικοί ανιχνευτές ακμών (Sobel, Canny) βασίζονται στην ιδέα της κλίσης. Εδώ τη βλέπετε στην απλούστερη μορφή της — σε μία διάσταση.

> 💡 **Στην κλινική πράξη:** Οι ακτινολόγοι σχεδιάζουν προφίλ για να μετρήσουν π.χ. το πλάτος αρτηρίας ή τη διάμετρο όγκου με ακρίβεια — όχι «με το μάτι».


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              INTENSITY PROFILE ANALYSIS                        ║
╚════════════════════════════════════════════════════════════════╝

Intensity profiles show how pixel values change along a line.
Useful for:
✓ Measuring edges and boundaries
✓ Detecting gradients
✓ Quality control
✓ Quantitative analysis
""")

def analyze_intensity_profiles(dicom_obj):
    """
    Analyze intensity profiles along different directions
    """
    
    img = dicom_obj.pixel_array
    rows, cols = img.shape
    
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    
    # Show image with profile lines
    axes[0, 0].imshow(img, cmap='gray')
    axes[0, 0].set_title('Image with Profile Lines', fontweight='bold')
    
    # Horizontal profile (middle row)
    mid_row = rows // 2
    axes[0, 0].axhline(y=mid_row, color='red', linestyle='--', linewidth=2, label='Horizontal')
    profile_h = img[mid_row, :]
    
    axes[0, 1].plot(profile_h, color='red', linewidth=2)
    axes[0, 1].set_title(f'Horizontal Profile (Row {mid_row})', fontweight='bold')
    axes[0, 1].set_xlabel('Column Position')
    axes[0, 1].set_ylabel('Intensity')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].fill_between(range(len(profile_h)), profile_h, alpha=0.3, color='red')
    
    # Vertical profile (middle column)
    mid_col = cols // 2
    axes[0, 0].axvline(x=mid_col, color='blue', linestyle='--', linewidth=2, label='Vertical')
    profile_v = img[:, mid_col]
    
    axes[1, 0].plot(profile_v, color='blue', linewidth=2)
    axes[1, 0].set_title(f'Vertical Profile (Column {mid_col})', fontweight='bold')
    axes[1, 0].set_xlabel('Row Position')
    axes[1, 0].set_ylabel('Intensity')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].fill_between(range(len(profile_v)), profile_v, alpha=0.3, color='blue')
    
    # Diagonal profile (top-left to bottom-right)
    diag_length = min(rows, cols)
    diag_indices = np.linspace(0, diag_length-1, diag_length).astype(int)
    profile_diag = img[diag_indices, diag_indices]
    
    axes[0, 0].plot([0, diag_length-1], [0, diag_length-1], 
                    color='green', linestyle='--', linewidth=2, label='Diagonal')
    axes[0, 0].legend()
    
    axes[1, 1].plot(profile_diag, color='green', linewidth=2)
    axes[1, 1].set_title('Diagonal Profile (TL to BR)', fontweight='bold')
    axes[1, 1].set_xlabel('Position along Diagonal')
    axes[1, 1].set_ylabel('Intensity')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].fill_between(range(len(profile_diag)), profile_diag, alpha=0.3, color='green')
    
    # Profile statistics comparison
    profiles_data = {
        'Horizontal': profile_h,
        'Vertical': profile_v,
        'Diagonal': profile_diag
    }
    
    profile_stats = []
    for name, profile in profiles_data.items():
        profile_stats.append({
            'Profile': name,
            'Mean': profile.mean(),
            'Std': profile.std(),
            'Min': profile.min(),
            'Max': profile.max(),
            'Range': profile.max() - profile.min()
        })
    
    profile_stats_df = pd.DataFrame(profile_stats)
    
    # Plot statistics comparison
    x = np.arange(len(profile_stats_df))
    width = 0.15
    
    axes[2, 0].bar(x - 2*width, profile_stats_df['Mean'], width, label='Mean', color='steelblue')
    axes[2, 0].bar(x - width, profile_stats_df['Std'], width, label='Std', color='orange')
    axes[2, 0].bar(x, profile_stats_df['Min'], width, label='Min', color='green')
    axes[2, 0].bar(x + width, profile_stats_df['Max'], width, label='Max', color='red')
    axes[2, 0].bar(x + 2*width, profile_stats_df['Range'], width, label='Range', color='purple')
    
    axes[2, 0].set_xlabel('Profile Type')
    axes[2, 0].set_ylabel('Value')
    axes[2, 0].set_title('Profile Statistics Comparison', fontweight='bold')
    axes[2, 0].set_xticks(x)
    axes[2, 0].set_xticklabels(profile_stats_df['Profile'])
    axes[2, 0].legend()
    axes[2, 0].grid(True, alpha=0.3, axis='y')
    
    # Gradient analysis (rate of change)
    gradient_h = np.gradient(profile_h)
    axes[2, 1].plot(gradient_h, color='darkred', linewidth=2)
    axes[2, 1].set_title('Horizontal Profile Gradient (Rate of Change)', fontweight='bold')
    axes[2, 1].set_xlabel('Column Position')
    axes[2, 1].set_ylabel('Gradient (Intensity Change)')
    axes[2, 1].grid(True, alpha=0.3)
    axes[2, 1].axhline(y=0, color='black', linestyle='-', linewidth=1)
    
    plt.tight_layout()
    plt.show()
    
    return profile_stats_df

# Perform profile analysis
profile_stats = analyze_intensity_profiles(dicom_data)
print("\n📊 Profile Statistics:")
display(profile_stats)


## Ενότητα 31 — Εξίσωση Ιστογράμματος (Histogram Equalization)

### Το πρόβλημα της χαμηλής αντίθεσης

Πολλές ιατρικές εικόνες χρησιμοποιούν **μόνο ένα μικρό κομμάτι** του διαθέσιμου εύρους έντασης. Αν π.χ. μια εικόνα 8-bit (0–255) χρησιμοποιεί μόνο τιμές 80–140, το ιστόγραμμα είναι «στριμωγμένο» στη μέση και η εικόνα φαίνεται **γκρίζα και άτονη** — χάνουμε λεπτομέρειες.

### Η ιδέα της εξίσωσης

Η **εξίσωση ιστογράμματος** είναι μια μη-γραμμική μεταμόρφωση που **ξαπλώνει** την κατανομή των εντάσεων ώστε να καλύψει όλο το διαθέσιμο εύρος. Στόχος: ένα ιστόγραμμα όσο πιο **ομοιόμορφο** γίνεται, που μεταφράζεται σε εικόνα με αυξημένη αντίθεση.

### Πώς δουλεύει — απλή μαθηματική διαίσθηση

Το κλειδί είναι η **CDF** (Cumulative Distribution Function): η αθροιστική κατανομή των εντάσεων. Τα βήματα είναι:

1. Υπολογίζουμε το ιστόγραμμα `h(i)`.
2. Υπολογίζουμε την CDF: `cdf(i) = Σ h(j) για j ≤ i`.
3. **Κανονικοποιούμε** την CDF στο εύρος [0, 255].
4. Χρησιμοποιούμε την κανονικοποιημένη CDF ως **lookup table**: κάθε αρχικό pixel τιμής `i` αντικαθίσταται με την τιμή `cdf'(i)`.

> 🧠 **Γιατί δουλεύει:** Η CDF, όταν κανονικοποιηθεί, είναι ο «τέλειος» μετασχηματισμός που μετατρέπει οποιαδήποτε κατανομή σε ομοιόμορφη.

### Τι θα δείτε στον κώδικα

Έξι υποδιαγράμματα που δείχνουν όλη τη διαδικασία:

| # | Τι δείχνει | Τι μαθαίνετε |
|---|------------|--------------|
| 1 | Αρχική εικόνα | Σημείο εκκίνησης |
| 2 | Αρχικό ιστόγραμμα | Πόσο «στριμωγμένο» ήταν |
| 3 | CDF | Η συνάρτηση μετασχηματισμού |
| 4 | Εικόνα μετά την εξίσωση | Αυξημένη αντίθεση |
| 5 | Νέο ιστόγραμμα | Πιο ομοιόμορφη κατανομή |
| 6 | Mapping function | Πώς απεικονίζονται οι παλιές → νέες τιμές |

### ⚠️ Προσοχή στην ιατρική χρήση

Η εξίσωση ιστογράμματος **αλλάζει** τις απόλυτες τιμές των pixel. Αυτό σημαίνει ότι:

- **ΔΕΝ** μπορεί να εφαρμοστεί σε CT αν θέλετε να διατηρήσετε τις **Hounsfield Units** (που έχουν φυσικό νόημα — π.χ. −1000 = αέρας, 0 = νερό).
- Είναι κατάλληλη για **οπτικοποίηση** ή ως pre-processing σε νευρωνικά δίκτυα φυσικών εικόνων.
- Σε MRI, μπορεί να χρησιμοποιηθεί προσεκτικά αν δεν σας ενδιαφέρουν απόλυτες εντάσεις.

> 📌 **Εναλλακτική:** Το **CLAHE** (Contrast-Limited Adaptive Histogram Equalization) εφαρμόζει εξίσωση τοπικά και είναι πιο φιλικό σε ιατρικές εικόνες. Δείτε το `cv2.createCLAHE()` αν θέλετε να πειραματιστείτε.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              HISTOGRAM EQUALIZATION                            ║
╚════════════════════════════════════════════════════════════════╝

Histogram equalization is a technique to improve image contrast
by redistributing pixel intensities to use the full dynamic range.

Applications:
✓ Enhancing low-contrast images
✓ Improving visibility of details
✓ Preprocessing for analysis
""")

def demonstrate_histogram_equalization(dicom_obj):
    """
    Demonstrate histogram equalization
    """
    
    img = dicom_obj.pixel_array.astype(float)
    
    # Normalize to 0-255 for equalization
    img_normalized = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
    
    # Compute histogram
    hist, bins = np.histogram(img_normalized.flatten(), bins=256, range=[0, 256])
    
    # Compute cumulative distribution function (CDF)
    cdf = hist.cumsum()
    cdf_normalized = cdf * hist.max() / cdf.max()  # Normalize for display
    
    # Equalization
    cdf_masked = np.ma.masked_equal(cdf, 0)
    cdf_masked = (cdf_masked - cdf_masked.min()) * 255 / (cdf_masked.max() - cdf_masked.min())
    cdf_final = np.ma.filled(cdf_masked, 0).astype('uint8')
    
    # Apply equalization
    img_equalized = cdf_final[img_normalized]
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Original image
    axes[0, 0].imshow(img_normalized, cmap='gray')
    axes[0, 0].set_title('Original Image', fontweight='bold', fontsize=12)
    axes[0, 0].axis('off')
    
    # Original histogram
    axes[0, 1].hist(img_normalized.flatten(), bins=256, range=[0, 256], 
                    color='steelblue', alpha=0.7, edgecolor='black')
    axes[0, 1].set_title('Original Histogram', fontweight='bold', fontsize=12)
    axes[0, 1].set_xlabel('Pixel Intensity')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].grid(True, alpha=0.3)
    
    # CDF
    axes[0, 2].plot(cdf_normalized, color='darkblue', linewidth=2)
    axes[0, 2].set_title('Cumulative Distribution Function', fontweight='bold', fontsize=12)
    axes[0, 2].set_xlabel('Pixel Intensity')
    axes[0, 2].set_ylabel('Cumulative Frequency')
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].set_xlim([0, 256])
    
    # Equalized image
    axes[1, 0].imshow(img_equalized, cmap='gray')
    axes[1, 0].set_title('Equalized Image', fontweight='bold', fontsize=12)
    axes[1, 0].axis('off')
    
    # Equalized histogram
    axes[1, 1].hist(img_equalized.flatten(), bins=256, range=[0, 256],
                    color='coral', alpha=0.7, edgecolor='black')
    axes[1, 1].set_title('Equalized Histogram', fontweight='bold', fontsize=12)
    axes[1, 1].set_xlabel('Pixel Intensity')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Comparison
    axes[1, 2].plot(cdf_final, color='red', linewidth=2, label='Mapping Function')
    axes[1, 2].set_title('Equalization Mapping Function', fontweight='bold', fontsize=12)
    axes[1, 2].set_xlabel('Input Intensity')
    axes[1, 2].set_ylabel('Output Intensity')
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].legend()
    axes[1, 2].set_xlim([0, 256])
    axes[1, 2].set_ylim([0, 256])
    
    plt.suptitle('Histogram Equalization Process', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
    
    # Statistics comparison
    print("\n📊 Comparison Statistics:")
    print("="*60)
    print(f"{'Metric':<30} {'Original':>12} {'Equalized':>12}")
    print("-"*60)
    print(f"{'Mean':<30} {img_normalized.mean():>12.2f} {img_equalized.mean():>12.2f}")
    print(f"{'Std Dev':<30} {img_normalized.std():>12.2f} {img_equalized.std():>12.2f}")
    print(f"{'Min':<30} {img_normalized.min():>12.2f} {img_equalized.min():>12.2f}")
    print(f"{'Max':<30} {img_normalized.max():>12.2f} {img_equalized.max():>12.2f}")
    print(f"{'Dynamic Range':<30} {img_normalized.ptp():>12.2f} {img_equalized.ptp():>12.2f}")
    print("="*60)

demonstrate_histogram_equalization(dicom_data)


## Ενότητα 32 — Ανάλυση Αντίθεσης και Φωτεινότητας

### Δύο διαφορετικές έννοιες — μη τις μπερδεύετε

Στην καθημερινή ομιλία λέμε «αυτή η εικόνα είναι σκοτεινή» ή «έχει χαμηλή αντίθεση» χωρίς πολλή σκέψη. Στην ποσοτική ανάλυση όμως είναι **δύο εντελώς ξεχωριστές μετρικές**:

| Έννοια | Τι μετρά | Σχέση με στατιστική |
|--------|----------|----------------------|
| **Φωτεινότητα** (brightness) | Συνολικό «επίπεδο» της εικόνας | Μέση τιμή των pixel |
| **Αντίθεση** (contrast) | Διαφορά μεταξύ φωτεινών και σκοτεινών περιοχών | Διασπορά των pixel |

> 💡 **Φανταστείτε:** Δύο εικόνες με την **ίδια** μέση τιμή. Η μία έχει pixel στην περιοχή 100–110 (πολύ χαμηλή αντίθεση — όλα μοιάζουν). Η άλλη έχει pixel σε όλη την κλίμακα 0–255 (υψηλή αντίθεση). Ίδια φωτεινότητα, εντελώς διαφορετική αντίθεση.

### Δύο τρόποι να μετρήσετε αντίθεση

Στον κώδικα υπολογίζονται και οι δύο γνωστές μετρικές:

#### 1. RMS contrast
$$
C_{\text{RMS}} = \sigma_I
$$
Απλή τυπική απόκλιση (στην κανονικοποιημένη εικόνα). **Δίνει βάρος σε όλα τα pixel** και είναι πιο αντιπροσωπευτική για φυσικές/ιατρικές εικόνες.

#### 2. Michelson contrast
$$
C_{\text{Michelson}} = \dfrac{I_{\max} - I_{\min}}{I_{\max} + I_{\min}}
$$
Λαμβάνει υπόψη **μόνο** τις ακραίες τιμές. Παραδοσιακά χρησιμοποιείται για **περιοδικά** μοτίβα (π.χ. test patterns σε οπτική). Σε ιατρικές εικόνες είναι ευαίσθητο σε outliers.

### Τι κάνει ο κώδικας

Δημιουργεί **πέντε εκδοχές** της ίδιας εικόνας για να δείτε **οπτικά** τη διαφορά:

1. **Αρχική** — αναφορά.
2. **Αυξημένη φωτεινότητα** — `image + 0.2`. Όλα τα pixel ανεβαίνουν.
3. **Μειωμένη φωτεινότητα** — `image - 0.2`. Όλα τα pixel πέφτουν.
4. **Αυξημένη αντίθεση** — `(image - 0.5) * 1.5 + 0.5`. Τα pixel «τραβιούνται» μακριά από το κέντρο.
5. **Μειωμένη αντίθεση** — `(image - 0.5) * 0.5 + 0.5`. Τα pixel «συμπιέζονται» γύρω από το κέντρο.

Στους τίτλους εμφανίζονται οι **νέες** τιμές brightness και contrast, ώστε να δείτε ποσοτικά πώς αλλάζουν.

> 📌 **Σύνδεση με κλινική πρακτική:** Αυτό ακριβώς που κάνει ένας ακτινολόγος όταν ρυθμίζει το **window/level** σε έναν σταθμό εργασίας — αλλάζει εικονικά την αντίθεση και τη φωτεινότητα **χωρίς να πειράζει τα δεδομένα**, για καλύτερη οπτική ερμηνεία.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║          CONTRAST AND BRIGHTNESS ANALYSIS                      ║
╚════════════════════════════════════════════════════════════════╝

Understanding contrast and brightness:
- Brightness: Overall intensity level (related to mean)
- Contrast: Difference between light and dark areas (related to std dev)
""")

def analyze_contrast_brightness(dicom_obj):
    """
    Analyze and visualize contrast and brightness
    """
    
    img = dicom_obj.pixel_array.astype(float)
    
    # Normalize
    img_norm = (img - img.min()) / (img.max() - img.min())
    
    # Calculate metrics
    brightness = img_norm.mean()
    contrast = img_norm.std()
    
    # Michelson contrast (for images with distinct bright and dark regions)
    max_val = img_norm.max()
    min_val = img_norm.min()
    michelson_contrast = (max_val - min_val) / (max_val + min_val) if (max_val + min_val) > 0 else 0
    
    # RMS contrast
    rms_contrast = img_norm.std()
    
    # Create variations
    # Increase brightness
    img_bright = np.clip(img_norm + 0.2, 0, 1)
    
    # Decrease brightness
    img_dark = np.clip(img_norm - 0.2, 0, 1)
    
    # Increase contrast
    img_high_contrast = np.clip((img_norm - 0.5) * 1.5 + 0.5, 0, 1)
    
    # Decrease contrast
    img_low_contrast = np.clip((img_norm - 0.5) * 0.5 + 0.5, 0, 1)
    
    # Visualize
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    images = [
        (img_norm, 'Original'),
        (img_bright, 'Increased Brightness'),
        (img_dark, 'Decreased Brightness'),
        (img_high_contrast, 'Increased Contrast'),
        (img_low_contrast, 'Decreased Contrast'),
    ]
    
    for idx, (image, title) in enumerate(images):
        row = idx // 3
        col = idx % 3
        
        axes[row, col].imshow(image, cmap='gray')
        
        # Calculate metrics for each variation
        b = image.mean()
        c = image.std()
        
        axes[row, col].set_title(f'{title}\nBrightness: {b:.3f}, Contrast: {c:.3f}',
                                fontweight='bold', fontsize=10)
        axes[row, col].axis('off')
    
    # Add metrics display
    axes[1, 2].axis('off')
    metrics_text = f"""
    CONTRAST METRICS
    ─────────────────────
    
    Brightness (Mean):
      {brightness:.4f}
    
    RMS Contrast (Std):
      {rms_contrast:.4f}
    
    Michelson Contrast:
      {michelson_contrast:.4f}
    
    Dynamic Range:
      {img_norm.ptp():.4f}
    
    ─────────────────────
    Interpretation:
    • Brightness: [0-1]
      0.5 = medium
    • Contrast: [0-∞]
      Higher = more contrast
    • Michelson: [0-1]
      1 = maximum contrast
    """
    
    axes[1, 2].text(0.1, 0.5, metrics_text, fontsize=10, family='monospace',
                   verticalalignment='center',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('Contrast and Brightness Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

analyze_contrast_brightness(dicom_data)


## Ενότητα 33 — Ανάλυση Θορύβου

### Τι είναι ο θόρυβος και από πού προέρχεται;

**Θόρυβος** είναι η ανεπιθύμητη τυχαία διακύμανση των τιμών των pixel που **δεν** σχετίζεται με την υποκείμενη ανατομία. Σε ιατρικές εικόνες προέρχεται από:

- **Θερμικός θόρυβος** — από τα ηλεκτρονικά κυκλώματα.
- **Κβαντικός θόρυβος (Poisson)** — εγγενής στη φύση των φωτονίων (CT, mammography).
- **Θόρυβος ψηφιοποίησης** — όρια ακρίβειας του ADC.
- **Κίνηση ασθενούς** — δεν είναι «θόρυβος» τεχνικά, αλλά εμφανίζεται με παρόμοιο τρόπο.

### Γιατί μας ενδιαφέρει;

- **Διαγνωστική ακρίβεια:** Πολύς θόρυβος καλύπτει μικρές βλάβες.
- **Αυτόματη ανάλυση:** Αλγόριθμοι segmentation, registration και AI παίρνουν χειρότερα αποτελέσματα σε θορυβώδεις εικόνες.
- **Quality assurance:** Σε κλινικά μηχανήματα, ο τακτικός έλεγχος SNR είναι μέρος του πρωτοκόλλου ποιότητας.

### Τρεις μέθοδοι εκτίμησης θορύβου

#### Μέθοδος 1: Τυπική απόκλιση σε ομοιογενή περιοχή
Σε μια **καθαρή** περιοχή (π.χ. αέρας στις γωνίες) η μόνη πηγή μεταβλητότητας είναι ο θόρυβος. Άρα:
$$
\sigma_{\text{noise}} \approx \sigma_{\text{corner}}
$$
Ο κώδικας παίρνει και τις τέσσερις γωνίες, υπολογίζει τη std σε καθεμιά, και κρατά τη μέση τιμή.

> ⚠️ **Προσοχή:** Αν τυχόν μια γωνία ΔΕΝ είναι φόντο (π.χ. ένας ώμος ασθενούς πέφτει στη γωνία), η μέθοδος δίνει υπερεκτίμηση. Γι' αυτό βλέπετε ότι κρατάμε και την **ελάχιστη** corner std — πιο ανθεκτική στην εξαίρεση.

#### Μέθοδος 2: MAD (Median Absolute Deviation)
$$
\text{MAD} = \text{median}(|x - \text{median}(x)|)
$$
Πλεονέκτημα: εξαιρετικά **ανθεκτική** σε outliers. Για κανονική κατανομή, η σχέση με την τυπική απόκλιση είναι:
$$
\sigma \approx 1{,}4826 \times \text{MAD}
$$

#### Μέθοδος 3: Gradient-based
Ο θόρυβος εμφανίζεται ως υψηλο-συχνοτικές διακυμάνσεις. Άρα η τυπική απόκλιση των διαφορών μεταξύ γειτονικών pixel (`np.diff`) σχετίζεται άμεσα με το επίπεδο θορύβου.

### Το SNR και η κλίμακα ποιότητας

$$
\text{SNR} = \dfrac{\mu_{\text{signal}}}{\sigma_{\text{noise}}}, \quad \text{SNR}_{\text{dB}} = 20 \log_{10}(\text{SNR})
$$

Σε **decibel** (dB) η κλίμακα γίνεται πιο διαισθητική:

| SNR (dB) | Ποιότητα |
|----------|----------|
| < 10 dB | Φτωχή — δύσκολα διαγνωστικά |
| 10–20 dB | Αποδεκτή για βασική απεικόνιση |
| 20–30 dB | Καλή — διαγνωστική ποιότητα |
| > 30 dB | Άριστη — ιδανική για ποσοτική ανάλυση |

### Τι παράγει ο κώδικας

Έξι οπτικοποιήσεις:

1. **Αρχική εικόνα** — αναφορά.
2. **Heatmap κλίσης (gradient magnitude)** — εκεί όπου εμφανίζεται **ταυτόχρονα** θόρυβος και πραγματικές ακμές.
3. **Ιστόγραμμα κλίσεων** σε λογαριθμική κλίμακα — η ουρά του υποδηλώνει επίπεδο θορύβου.
4. **Εικόνα με σημειωμένες τις γωνίες** που χρησιμοποιήθηκαν για εκτίμηση.
5. **Πίνακας μετρικών** — όλες οι μετρήσεις σε ένα μέρος.
6. **Διάγραμμα ποιότητας** — που πέφτει η εικόνα μας στην παραπάνω κλίμακα.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                    NOISE ANALYSIS                              ║
╚════════════════════════════════════════════════════════════════╝

Noise in medical images affects:
✓ Image quality
✓ Diagnostic accuracy
✓ Automated analysis algorithms

Common types:
- Gaussian noise
- Salt & pepper noise
- Poisson noise
""")

def estimate_noise_level(dicom_obj, method='std_uniform_region'):
    """
    Estimate noise level in the image
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object
    method : str
        Method for noise estimation
    
    Returns:
    --------
    dict : Noise metrics
    """
    
    img = dicom_obj.pixel_array.astype(float)
    
    noise_metrics = {}
    
    # Method 1: Standard deviation in a uniform region (background)
    # Assume corners are relatively uniform background
    corner_size = min(img.shape) // 10
    corners = [
        img[:corner_size, :corner_size],  # Top-left
        img[:corner_size, -corner_size:],  # Top-right
        img[-corner_size:, :corner_size],  # Bottom-left
        img[-corner_size:, -corner_size:]  # Bottom-right
    ]
    
    corner_stds = [corner.std() for corner in corners]
    noise_metrics['corner_std_mean'] = np.mean(corner_stds)
    noise_metrics['corner_std_min'] = np.min(corner_stds)
    
    # Method 2: Median Absolute Deviation (MAD) - robust to outliers
    median = np.median(img)
    mad = np.median(np.abs(img - median))
    noise_metrics['mad'] = mad
    noise_metrics['estimated_std_from_mad'] = 1.4826 * mad  # Conversion factor for normal distribution
    
    # Method 3: Gradient-based noise estimation
    # Noise often shows up in high-frequency components
    grad_x = np.abs(np.diff(img, axis=1))
    grad_y = np.abs(np.diff(img, axis=0))
    noise_metrics['gradient_std_x'] = grad_x.std()
    noise_metrics['gradient_std_y'] = grad_y.std()
    
    # Signal-to-Noise Ratio
    signal = img.mean()
    noise = noise_metrics['estimated_std_from_mad']
    noise_metrics['snr'] = signal / noise if noise > 0 else float('inf')
    noise_metrics['snr_db'] = 20 * np.log10(noise_metrics['snr']) if noise_metrics['snr'] > 0 else float('inf')
    
    return noise_metrics

def visualize_noise_analysis(dicom_obj):
    """
    Visualize noise characteristics
    """
    
    img = dicom_obj.pixel_array.astype(float)
    
    # Estimate noise
    noise_metrics = estimate_noise_level(dicom_obj)
    
    # Compute gradients
    grad_x = np.diff(img, axis=1)
    grad_y = np.diff(img, axis=0)
    grad_magnitude = np.sqrt(grad_x[:, :-1]**2 + grad_y[:-1, :]**2)
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Original image
    axes[0, 0].imshow(img, cmap='gray')
    axes[0, 0].set_title('Original Image', fontweight='bold')
    axes[0, 0].axis('off')
    
    # Gradient magnitude (edge/noise)
    im1 = axes[0, 1].imshow(grad_magnitude, cmap='hot')
    axes[0, 1].set_title('Gradient Magnitude\n(Edges + Noise)', fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1])
    
    # Histogram of gradients
    axes[0, 2].hist(grad_magnitude.flatten(), bins=50, color='orange', alpha=0.7, edgecolor='black')
    axes[0, 2].set_title('Gradient Histogram', fontweight='bold')
    axes[0, 2].set_xlabel('Gradient Magnitude')
    axes[0, 2].set_ylabel('Frequency')
    axes[0, 2].set_yscale('log')
    axes[0, 2].grid(True, alpha=0.3)
    
    # Corner regions for noise estimation
    corner_size = min(img.shape) // 10
    img_corners = img.copy()
    
    # Mark corners
    from matplotlib.patches import Rectangle
    axes[1, 0].imshow(img, cmap='gray')
    corners_coords = [
        (0, 0), (img.shape[1]-corner_size, 0),
        (0, img.shape[0]-corner_size), (img.shape[1]-corner_size, img.shape[0]-corner_size)
    ]
    
    for x, y in corners_coords:
        rect = Rectangle((x, y), corner_size, corner_size, 
                        linewidth=2, edgecolor='red', facecolor='none')
        axes[1, 0].add_patch(rect)
    
    axes[1, 0].set_title('Corner Regions\n(for noise estimation)', fontweight='bold')
    axes[1, 0].axis('off')
    
    # Noise metrics display
    axes[1, 1].axis('off')
    metrics_text = f"""
    NOISE METRICS
    ══════════════════════════
    
    Corner STD (mean):
      {noise_metrics['corner_std_mean']:.2f}
    
    MAD (Median Abs Dev):
      {noise_metrics['mad']:.2f}
    
    Estimated Noise (σ):
      {noise_metrics['estimated_std_from_mad']:.2f}
    
    Gradient STD (X):
      {noise_metrics['gradient_std_x']:.2f}
    
    Gradient STD (Y):
      {noise_metrics['gradient_std_y']:.2f}
    
    Signal-to-Noise Ratio:
      {noise_metrics['snr']:.2f}
    
    SNR (dB):
      {noise_metrics['snr_db']:.2f} dB
    
    ══════════════════════════
    Higher SNR = Better quality
    Typical good SNR > 20 dB
    """
    
    axes[1, 1].text(0.1, 0.5, metrics_text, fontsize=9, family='monospace',
                   verticalalignment='center',
                   bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    
    # SNR visualization
    snr_categories = ['Poor\n(<10 dB)', 'Fair\n(10-20 dB)', 'Good\n(20-30 dB)', 'Excellent\n(>30 dB)']
    snr_values = [10, 20, 30, 40]
    colors = ['red', 'orange', 'yellow', 'green']
    
    bars = axes[1, 2].barh(snr_categories, snr_values, color=colors, alpha=0.6, edgecolor='black')
    
    # Mark current SNR
    current_snr = noise_metrics['snr_db']
    if current_snr < 10:
        marker_pos = 0
    elif current_snr < 20:
        marker_pos = 1
    elif current_snr < 30:
        marker_pos = 2
    else:
        marker_pos = 3
    
    axes[1, 2].plot([current_snr], [marker_pos], 'r*', markersize=20, 
                   label=f'Current: {current_snr:.1f} dB')
    
    axes[1, 2].set_xlabel('SNR (dB)')
    axes[1, 2].set_title('SNR Quality Assessment', fontweight='bold')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3, axis='x')
    
    plt.suptitle('Comprehensive Noise Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_noise_analysis(dicom_data)


## Ενότητα 34 — Σύνταξη Ολοκληρωμένης Στατιστικής Αναφοράς

### Από τα κομμάτια στο σύνολο

Μέχρι εδώ, μάθαμε **μεμονωμένες** τεχνικές: ιστογράμματα, στατιστικά, χωρική ανάλυση, θόρυβο. Σε πραγματική κλινική ή ερευνητική χρήση πρέπει να τα συνθέσουμε όλα σε **μία αναφορά** που:

- Συγκεντρώνει τα μεταδεδομένα του ασθενούς και της εξέτασης.
- Δίνει στατιστικά χαρακτηριστικά της εικόνας.
- Αξιολογεί την ποιότητα.
- Παρουσιάζει τα παραπάνω σε αναγνώσιμη μορφή για διαφορετικά ακροατήρια (γιατρός, ερευνητής, μηχανικός).

### Τι κάνει η `generate_complete_dicom_report`

Επιστρέφει ένα **nested dictionary** με τέσσερα τμήματα:

1. **`metadata`** — Modality, Patient ID, Study Date, μέγεθος εικόνας, pixel spacing, slice thickness.
2. **`basic_stats`** — Όλα τα στατιστικά της Ενότητας 27.
3. **`histogram_stats`** — Skewness, kurtosis, CV.
4. **`quality_metrics`** — SNR, εκτίμηση θορύβου από MAD.
5. **`spatial_stats`** — Brightness, RMS contrast, dynamic range.

Ταυτόχρονα παράγει **ένα ολοκληρωμένο γραφικό dashboard** με:

- Την εικόνα.
- Το ιστόγραμμα με σημειωμένη μέση τιμή και διάμεσο.
- Box plot.
- Πίνακες μεταδεδομένων, στατιστικών και ποιότητας.
- Προφίλ έντασης μέσης γραμμής.

> 💡 **Καλή πρακτική κώδικα:** Παρατηρήστε ότι η συνάρτηση δέχεται προαιρετικό `save_path`. Έτσι η ίδια λειτουργία μπορεί να (α) εμφανίσει την αναφορά στο Jupyter για εξερεύνηση, ή (β) να την αποθηκεύσει σε αρχείο για παράδοση. Αυτό λέγεται **separation of concerns** και είναι θεμελιώδες σε επαγγελματικό κώδικα.

### Γιατί μας νοιάζει το JSON output

Στο τέλος του κελιού βλέπετε `print(json.dumps(report_data, indent=2))`. Γιατί JSON;

- **Διαλειτουργικότητα:** Άλλα προγράμματα μπορούν να διαβάσουν το JSON και να επεξεργαστούν τα αποτελέσματα.
- **Αρχειοθέτηση:** Σε μελέτες με 1000+ εικόνες, αποθηκεύουμε ένα JSON ανά εικόνα και τα συγκεντρώνουμε σε database.
- **Reproducibility:** Ο επόμενος ερευνητής βλέπει ακριβώς τι μετρήθηκε.

> 🧠 **Σκέψη για τον φοιτητή:** Σε κάθε δικό σας project, σχεδιάζετε από την αρχή «τι αναφορά θα παράγω;». Αυτό συχνά διαμορφώνει σωστά και τον υπόλοιπο κώδικα.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║          GENERATING COMPLETE STATISTICAL REPORT                ║
╚════════════════════════════════════════════════════════════════╝
""")

def generate_complete_dicom_report(dicom_obj, save_path=None):
    """
    Generate a complete statistical and quality report for a DICOM image
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object to analyze
    save_path : str, optional
        Path to save the report
    
    Returns:
    --------
    dict : Complete report data
    """
    
    img = dicom_obj.pixel_array
    
    # Collect all analyses
    report = {
        'metadata': {},
        'basic_stats': {},
        'histogram_stats': {},
        'quality_metrics': {},
        'spatial_stats': {}
    }
    
    # Metadata
    report['metadata'] = {
        'Modality': dicom_obj.get('Modality', 'N/A'),
        'Patient_ID': dicom_obj.get('PatientID', 'N/A'),
        'Study_Date': dicom_obj.get('StudyDate', 'N/A'),
        'Image_Size': f"{dicom_obj.Rows} x {dicom_obj.Columns}",
        'Pixel_Spacing': str(dicom_obj.get('PixelSpacing', 'N/A')),
        'Slice_Thickness': str(dicom_obj.get('SliceThickness', 'N/A'))
    }
    
    # Basic statistics
    flat_img = img.flatten()
    report['basic_stats'] = {
        'Mean': float(flat_img.mean()),
        'Median': float(np.median(flat_img)),
        'Std_Dev': float(flat_img.std()),
        'Variance': float(flat_img.var()),
        'Min': float(flat_img.min()),
        'Max': float(flat_img.max()),
        'Range': float(flat_img.ptp()),
        'Q1': float(np.percentile(flat_img, 25)),
        'Q3': float(np.percentile(flat_img, 75)),
        'IQR': float(np.percentile(flat_img, 75) - np.percentile(flat_img, 25))
    }
    
    # Histogram statistics
    from scipy import stats as scipy_stats
    report['histogram_stats'] = {
        'Skewness': float(scipy_stats.skew(flat_img)),
        'Kurtosis': float(scipy_stats.kurtosis(flat_img)),
        'CV_Percent': float((flat_img.std() / flat_img.mean()) * 100) if flat_img.mean() != 0 else 0,
    }
    
    # Quality metrics
    noise_metrics = estimate_noise_level(dicom_obj)
    report['quality_metrics'] = {
        'SNR': float(noise_metrics['snr']),
        'SNR_dB': float(noise_metrics['snr_db']),
        'Estimated_Noise_Level': float(noise_metrics['estimated_std_from_mad']),
        'MAD': float(noise_metrics['mad'])
    }
    
    # Spatial statistics
    img_norm = (img - img.min()) / (img.max() - img.min())
    report['spatial_stats'] = {
        'Brightness': float(img_norm.mean()),
        'Contrast_RMS': float(img_norm.std()),
        'Dynamic_Range': float(img_norm.ptp())
    }
    
    # Create comprehensive visualization
    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.3)
    
    # 1. Main image
    ax1 = fig.add_subplot(gs[0:2, 0:2])
    ax1.imshow(img, cmap='gray')
    ax1.set_title(f'DICOM Image - {report["metadata"]["Modality"]}', 
                 fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    # 2. Histogram
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.hist(flat_img, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax2.axvline(report['basic_stats']['Mean'], color='red', linestyle='--', 
               linewidth=2, label=f"Mean: {report['basic_stats']['Mean']:.1f}")
    ax2.axvline(report['basic_stats']['Median'], color='green', linestyle='--',
               linewidth=2, label=f"Median: {report['basic_stats']['Median']:.1f}")
    ax2.set_title('Intensity Distribution', fontsize=10, fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    # 3. Box plot
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.boxplot(flat_img, vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax3.set_title('Box Plot', fontsize=10, fontweight='bold')
    ax3.set_ylabel('Intensity')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Metadata table
    ax4 = fig.add_subplot(gs[2, 0])
    ax4.axis('off')
    metadata_text = "METADATA\n" + "─"*30 + "\n"
    for key, value in report['metadata'].items():
        metadata_text += f"{key.replace('_', ' ')}: {value}\n"
    ax4.text(0.1, 0.5, metadata_text, fontsize=9, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
    
    # 5. Basic statistics table
    ax5 = fig.add_subplot(gs[2, 1])
    ax5.axis('off')
    stats_text = "BASIC STATISTICS\n" + "─"*30 + "\n"
    for key, value in report['basic_stats'].items():
        stats_text += f"{key.replace('_', ' ')}: {value:.2f}\n"
    ax5.text(0.1, 0.5, stats_text, fontsize=9, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    
    # 6. Quality metrics table
    ax6 = fig.add_subplot(gs[2, 2])
    ax6.axis('off')
    quality_text = "QUALITY METRICS\n" + "─"*30 + "\n"
    for key, value in report['quality_metrics'].items():
        quality_text += f"{key.replace('_', ' ')}: {value:.2f}\n"
    ax6.text(0.1, 0.5, quality_text, fontsize=9, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
    
    # 7. Intensity profile
    ax7 = fig.add_subplot(gs[3, :])
    mid_row = img.shape[0] // 2
    profile = img[mid_row, :]
    ax7.plot(profile, linewidth=2, color='darkblue')
    ax7.fill_between(range(len(profile)), profile, alpha=0.3)
    ax7.set_title(f'Intensity Profile (Row {mid_row})', fontsize=10, fontweight='bold')
    ax7.set_xlabel('Column Position')
    ax7.set_ylabel('Intensity')
    ax7.grid(True, alpha=0.3)
    
    # Overall title
    plt.suptitle('COMPREHENSIVE DICOM IMAGE ANALYSIS REPORT', 
                fontsize=18, fontweight='bold', y=0.98)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Report saved to: {save_path}")
    
    plt.show()
    
    return report

# Generate complete report
print("\n" + "="*70)
print("GENERATING COMPREHENSIVE REPORT...")
print("="*70)

report_data = generate_complete_dicom_report(dicom_data)

# Print summary
print("\n📋 REPORT SUMMARY:")
print(json.dumps(report_data, indent=2))


## Ενότητα 35 — Ασκήσεις Πρακτικής Εξάσκησης

### Γιατί ασκήσεις;

Δεν μαθαίνεται η ανάλυση εικόνων διαβάζοντας. Μαθαίνεται **τρέχοντας κώδικα, σπάζοντάς τον, και διορθώνοντας τα σφάλματα**. Οι παρακάτω οκτώ ασκήσεις είναι σχεδιασμένες ώστε να καλύπτουν διαφορετικές πτυχές αυτού που μάθαμε:

| # | Άσκηση | Τι εξασκεί |
|---|--------|-----------|
| 1 | Στατιστική Ανάλυση | Βασική ποσοτικοποίηση + έλεγχος κανονικότητας |
| 2 | Manipulation Ιστογράμματος | Επίδραση παραμέτρων στην ερμηνεία |
| 3 | Region Analysis | Χωρική ανάλυση + heatmaps |
| 4 | Noise Assessment | Κριτική σύγκριση μεθόδων |
| 5 | Profile Analysis | Μέτρηση σε φυσικές μονάδες (mm) |
| 6 | Comparative Analysis | Σύγκριση modalities |
| 7 | Quality Control | Σχεδιασμός pipeline |
| 8 | Advanced Visualization | Σύνθεση παρουσίασης |

### Συμβουλές για να δουλέψετε σωστά

> 📌 **Πάντοτε:**
> 1. **Επικυρώνετε τα δεδομένα εισόδου** — το πρώτο πράγμα που πρέπει να ελέγχετε είναι αν το DICOM φορτώθηκε σωστά (`dicom_obj.pixel_array.shape`, `dicom_obj.Modality`).
> 2. **Σχολιάζετε τον κώδικα** — γράψτε τι κάνετε ΚΑΙ γιατί. Σε έναν μήνα δεν θα το θυμάστε.
> 3. **Συγκρίνετε με αναμενόμενες τιμές** — π.χ. αν η μέση τιμή ενός CT φόντου σας βγει 500, κάτι πάει στραβά (φόντο = αέρας ≈ −1000 HU).
> 4. **Οπτικοποιείτε ενδιάμεσα βήματα** — μην αφήνετε τίποτα ως «μαύρο κουτί».
> 5. **Σκέφτεστε την κλινική σημασία** — δεν φτιάχνουμε γραφήματα για τα γραφήματα. Πάντα ρωτάτε «τι θα έλεγε ένας γιατρός βλέποντας αυτό;».

### Πώς να παραδώσετε

Σε κάθε άσκηση ζητείται:
- **Λειτουργικός κώδικας** σε notebook ή script.
- **Σχόλια** που εξηγούν τη λογική.
- **Ποιοτική ερμηνεία** των αποτελεσμάτων (όχι μόνο αριθμοί — τι σημαίνουν).

> 💡 Η καλύτερη παράδοση είναι αυτή που, αν τη δει κάποιος που δεν ξέρει DICOM, θα την καταλάβει πλήρως.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              ENHANCED PRACTICE EXERCISES                       ║
╚════════════════════════════════════════════════════════════════╝

📝 EXERCISE 1: Statistical Analysis
   Task: Load a DICOM image and:
   a) Calculate all central tendency measures (mean, median, mode)
   b) Calculate all dispersion measures (std, variance, range, IQR)
   c) Determine if the distribution is normal (using Q-Q plot)
   d) Identify and explain any outliers

📝 EXERCISE 2: Histogram Manipulation
   Task: 
   a) Create histograms with different bin sizes (10, 50, 100, 200)
   b) Compare how bin size affects interpretation
   c) Apply histogram equalization
   d) Compare before/after statistics

📝 EXERCISE 3: Region Analysis
   Task:
   a) Divide image into 4x4 grid
   b) Calculate mean and std for each region
   c) Create heatmaps showing spatial variation
   d) Identify regions with highest/lowest contrast

📝 EXERCISE 4: Noise Assessment
   Task:
   a) Estimate noise level using multiple methods
   b) Calculate SNR
   c) Compare corner regions for noise uniformity
   d) Suggest if image quality is adequate

📝 EXERCISE 5: Profile Analysis
   Task:
   a) Extract horizontal, vertical, and diagonal profiles
   b) Calculate gradients (rate of change)
   c) Identify edges and transitions
   d) Measure distances in physical units (mm)

📝 EXERCISE 6: Comparative Analysis
   Task:
   a) Load 3 different DICOM images (different modalities if possible)
   b) Create comparative statistics table
   c) Generate histograms side-by-side
   d) Explain differences based on modality

📝 EXERCISE 7: Quality Control
   Task:
   a) Create a quality control pipeline
   b) Check for: artifacts, proper contrast, noise levels
   c) Generate automatic quality report
   d) Flag images that fail quality criteria

📝 EXERCISE 8: Advanced Visualization
   Task:
   a) Create a dashboard with: image, histogram, statistics, profiles
   b) Add interactive elements (if possible)
   c) Include both spatial and statistical views
   d) Export as a report

💡 TIPS FOR EXERCISES:
   - Always validate your inputs
   - Document your code with comments
   - Compare results with expected values
   - Visualize intermediate steps
   - Think about clinical relevance
""")


## Ενότητα 36 — Συνοπτικός Οδηγός Αναφοράς

### Τι είναι αυτή η ενότητα;

Ένα **cheat sheet** που μπορείτε να έχετε δίπλα σας όταν προγραμματίζετε. Μαζευτήκαμε σε ένα μέρος όλες τις βασικές συναρτήσεις που χρησιμοποιήσαμε — οργανωμένες ανά κατηγορία.

### Πώς να το χρησιμοποιείτε

- **Πρώτη γραμμή άμυνας:** Όταν δεν θυμάστε πώς υπολογίζεται κάτι, κοιτάτε εδώ πριν ψάξετε στο Google.
- **Reference για ασκήσεις και projects:** Αντιγράφετε τα μοτίβα και τα προσαρμόζετε.
- **Σύνδεση εννοιών με κώδικα:** Κάθε στατιστική έννοια (μέση τιμή, IQR, kurtosis...) έχει τη δική της κλήση συνάρτησης.

### Οι κατηγορίες

| Κατηγορία | Τι περιλαμβάνει |
|-----------|------------------|
| **Central Tendency** | mean, median, mode |
| **Dispersion** | std, variance, range, IQR, CV |
| **Percentiles** | quartiles και custom percentiles |
| **Distribution Shape** | skewness, kurtosis |
| **Histogram** | basic, normalized, cumulative |
| **Image Quality** | SNR (linear & dB), contrast, brightness |
| **DICOM Access** | direct, safe (`.get`), tag access, pixel access |
| **Common Operations** | normalize, window, threshold, gradient |

### Επόμενα βήματα μετά αυτό το μάθημα

Έχετε πλέον τις βάσεις για να εξερευνήσετε:

1. **Segmentation** — αυτόματη οριοθέτηση οργάνων ή βλαβών.
2. **Registration** — ευθυγράμμιση εικόνων διαφορετικών εξετάσεων.
3. **Radiomics** — εξαγωγή εκατοντάδων χαρακτηριστικών για ML.
4. **Deep Learning σε ιατρικές εικόνες** — CNN, U-Net, transformers.
5. **Multi-modal ανάλυση** — συνδυασμός CT/MRI/PET.

> 🎓 **Τελική σκέψη:** Η ανάλυση ιατρικών εικόνων είναι μια από τις περιοχές της AI με μεγαλύτερο **κλινικό και ηθικό βάρος**. Αυτό που σήμερα μαθαίνετε ως «παιχνίδι με αριθμούς» μπορεί αύριο να επηρεάζει διαγνώσεις. Πάντα να σκέφτεστε: σωστά δεδομένα, σωστές παραδοχές, διαφανής αναφορά.

🏥 **Καλή ανάλυση!**


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                  QUICK REFERENCE GUIDE                         ║
╚════════════════════════════════════════════════════════════════╝

📚 STATISTICAL MEASURES QUICK REFERENCE:

CENTRAL TENDENCY:
├─ Mean:     np.mean(array)
├─ Median:   np.median(array)
└─ Mode:     scipy.stats.mode(array)

DISPERSION:
├─ Std Dev:  np.std(array)
├─ Variance: np.var(array)
├─ Range:    np.ptp(array)  # peak-to-peak
├─ IQR:      Q3 - Q1
└─ CV:       (std/mean) * 100

PERCENTILES:
├─ Quartiles: np.percentile(array, [25, 50, 75])
├─ Custom:    np.percentile(array, p)
└─ Quantile:  np.quantile(array, q)

DISTRIBUTION SHAPE:
├─ Skewness: scipy.stats.skew(array)
└─ Kurtosis: scipy.stats.kurtosis(array)

HISTOGRAM:
├─ Basic:    plt.hist(array, bins=50)
├─ Normed:   plt.hist(array, density=True)
└─ Cumul:    np.cumsum(counts)

IMAGE QUALITY:
├─ SNR:      mean / std
├─ SNR (dB): 20 * log10(SNR)
├─ Contrast: std of normalized image
└─ Bright:   mean of normalized image

DICOM ACCESS:
├─ Direct:   dicom_obj.TagName
├─ Safe:     dicom_obj.get('TagName', default)
├─ Tag num:  dicom_obj[0x0008, 0x0060].value
└─ Pixel:    dicom_obj.pixel_array

COMMON OPERATIONS:
├─ Normalize:  (img - min) / (max - min)
├─ Window:     np.clip(img, wc-ww/2, wc+ww/2)
├─ Threshold:  img > threshold
└─ Gradient:   np.gradient(img)
""")

print("""
╔════════════════════════════════════════════════════════════════╗
║                    TUTORIAL COMPLETE!                          ║
╚════════════════════════════════════════════════════════════════╝

🎉 Congratulations! You now have comprehensive knowledge of:

✓ DICOM image fundamentals
✓ Statistical analysis techniques
✓ Histogram interpretation and manipulation
✓ Quality assessment methods
✓ Noise analysis
✓ Contrast and brightness evaluation
✓ Region-based analysis
✓ Profile analysis
✓ Complete reporting

🚀 NEXT STEPS:
1. Practice with real DICOM datasets
2. Experiment with different modalities
3. Build your own analysis tools
4. Explore advanced topics (segmentation, registration, etc.)
5. Apply these techniques to research projects

📖 Keep this notebook as a reference for future work!

Happy analyzing! 🏥💻📊
""")
